# Regex Performance

## Overview
Regex queries are expensive because they require evaluating a pattern against every row, preventing standard B-tree index use.

**Performance hierarchy (fastest → slowest):**

| Pattern | Index strategy |
|---|---|
| `col = 'exact'` | B-tree seek |
| `col IN ('a','b','c')` | B-tree scan — use instead of alternation regex |
| `col LIKE 'prefix%'` | B-tree range — use instead of leading-anchor regex |
| `col ~ '^prefix'` | B-tree range (PostgreSQL, leading literal only) |
| `col ~ 'middle'` | pg_trgm GIN index |
| `col LIKE '%middle%'` | pg_trgm GIN index |
| `col ~ '(a\|b\|c)'` | Sequential scan — use IN() |
| `col ~ 'complex'` | Sequential scan — pre-filter + pg_trgm |

**pg_trgm** breaks strings into 3-character substrings stored in a GIN index. A regex query finds rows whose trigrams overlap the pattern, then verifies only those rows — dramatic speedup for free-text columns.

---

In [ ]:
import sqlite3, re, random, string

def make_conn():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.create_function("regexp", 2,
        lambda p, s: bool(re.search(p, s or "", re.IGNORECASE)))
    return conn

conn = make_conn()
conn.execute("""
    CREATE TABLE patients (
        patient_id TEXT PRIMARY KEY,
        full_name  TEXT NOT NULL,
        mrn        TEXT,
        province   TEXT
    )
""")
random.seed(42)
provinces = ['NB','ON','BC','QC','NS','AB','MB','SK']
rows = []
for i in range(1, 1001):
    pid  = f'P{i:04d}'
    name = (''.join(random.choices(string.ascii_lowercase, k=6)).capitalize()
            + ' '
            + ''.join(random.choices(string.ascii_lowercase, k=6)).capitalize())
    mrn  = (f'MRN-{random.randint(100000,999999):06d}'
            if random.random() > 0.05 else 'BAD-MRN')
    rows.append((pid, name, mrn, random.choice(provinces)))
conn.executemany("INSERT INTO patients VALUES (?,?,?,?)", rows)
conn.commit()

total   = conn.execute("SELECT COUNT(*) FROM patients").fetchone()[0]
invalid = conn.execute(
    "SELECT COUNT(*) FROM patients WHERE NOT mrn REGEXP '^MRN-[0-9]{6}$'"
).fetchone()[0]
print(f"{total} patients loaded, {invalid} invalid MRNs")


---
## Index usage and timing

In [ ]:
import time

print("=== Regex scan vs equality scan vs indexed equality ===")
print()

t0 = time.perf_counter()
n_regex = conn.execute(
    "SELECT COUNT(*) FROM patients WHERE province REGEXP '^ON$'"
).fetchone()[0]
regex_ms = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
n_eq = conn.execute(
    "SELECT COUNT(*) FROM patients WHERE province = 'ON'"
).fetchone()[0]
eq_ms = (time.perf_counter() - t0) * 1000

conn.execute("CREATE INDEX idx_province ON patients(province)")

t0 = time.perf_counter()
n_idx = conn.execute(
    "SELECT COUNT(*) FROM patients WHERE province = 'ON'"
).fetchone()[0]
idx_ms = (time.perf_counter() - t0) * 1000

print(f"  REGEXP scan:            {n_regex:4d} rows  {regex_ms:.3f}ms  (full table scan)")
print(f"  equality (no index):    {n_eq:4d} rows  {eq_ms:.3f}ms  (full table scan)")
print(f"  equality (with index):  {n_idx:4d} rows  {idx_ms:.3f}ms  (index seek)")
print()

print("Index usage rules (PostgreSQL):")
rules = [
    ("col = 'exact'",          "B-tree seek       -- always indexed"),
    ("col IN ('a','b','c')",   "B-tree scan       -- use instead of alternation regex"),
    ("col LIKE 'prefix%'",     "B-tree range      -- prefix anchored"),
    ("col ~ '^prefix'",        "B-tree range (PG) -- leading anchor only"),
    ("col ~ 'middle'",         "pg_trgm GIN       -- needs extension"),
    ("col LIKE '%middle%'",    "pg_trgm GIN       -- needs extension"),
    ("col ~ '(a|b|c)'",       "Sequential scan   -- use IN() instead"),
    ("col ~ 'complex'",        "Sequential scan   -- consider pre-filter + pg_trgm"),
]
print(f"  {'Pattern':<32s}  Index strategy")
print("  " + "-"*60)
for pat, strategy in rules:
    print(f"  {pat:<32s}  {strategy}")


---
## pg_trgm: trigram indexes

In [ ]:
print("=== pg_trgm: trigram indexes for regex and fuzzy search ===")
print()

word = "Lisinopril"
trigrams = [word[i:i+3] for i in range(len(word)-2)]
print(f"Trigrams of '{word}': {trigrams}")
print("pg_trgm stores all trigrams in a GIN index.")
print("A regex/ILIKE query finds rows sharing enough trigrams, then verifies.")
print()

print("""PostgreSQL setup and usage:

  -- Enable (once per database, requires superuser):
  CREATE EXTENSION IF NOT EXISTS pg_trgm;

  -- GIN trigram index on a free-text column:
  CREATE INDEX idx_notes_trgm
      ON clinical_notes USING GIN (note_text gin_trgm_ops);

  -- These queries now use the index:
  SELECT * FROM clinical_notes WHERE note_text ~*  'lisinopril';
  SELECT * FROM clinical_notes WHERE note_text ILIKE '%lisinopril%';

  -- Fuzzy patient name matching (handles misspellings):
  SELECT full_name, similarity(full_name, 'Ngatta') AS sim
  FROM   patients
  WHERE  full_name % 'Ngatta'   -- % operator: similarity > threshold (default 0.3)
  ORDER  BY sim DESC LIMIT 5;

  -- Verify index is used:
  EXPLAIN (ANALYZE, BUFFERS)
  SELECT * FROM clinical_notes WHERE note_text ~* 'lisinopril';
  -- With GIN:    Bitmap Index Scan on idx_notes_trgm
  -- Without:     Seq Scan on clinical_notes
""")

print("When to use pg_trgm:")
cases = [
    ("Regex / ILIKE on large free-text columns",   "GIN trigram index -- large speedup"),
    ("Fuzzy patient or drug name matching",         "similarity() + % operator"),
    ("Drug name autocomplete API",                  "GIN on drug_name + ILIKE"),
    ("ICD code prefix search",                      "B-tree is better for pure prefix"),
    ("Exact match queries",                         "B-tree always faster than trigram"),
    ("Small tables (< 10k rows)",                   "Seq scan fine; skip the index"),
]
for case, guidance in cases:
    print(f"  {case:<44s}  {guidance}")


---
## Strategy summary

In [ ]:
print("=== Performance strategies: ranked best to worst for regex queries ===")
print()

strategies = [
    (1, "Replace regex with equality or IN()",
     "province IN ('NB','NS','PE','NL')  vs  ~ '^(NB|NS|PE|NL)$'",
     "B-tree index; no regex evaluation at all"),
    (2, "Replace regex with LIKE for prefix",
     "mrn LIKE 'MRN-%'  vs  mrn ~ '^MRN-'",
     "B-tree range scan; faster than regex on large tables"),
    (3, "Pre-filter with an indexed column first",
     "WHERE province = 'ON' AND mrn ~ '^MRN-[0-9]{6}$'",
     "Index on province reduces rows; regex only runs on survivors"),
    (4, "Add GIN trigram index for regex on free text",
     "CREATE INDEX USING GIN (note_text gin_trgm_ops)",
     "Trigram pre-filter + exact regex verify; much faster than seq scan"),
    (5, "Functional index for repeated computed boolean",
     "CREATE INDEX ON patients ((mrn ~ '^MRN-[0-9]{6}$'))",
     "Stores boolean; WHERE (mrn ~ pattern) = true uses index"),
    (6, "Materialise a cleaned/validated column",
     "ALTER TABLE medications ADD COLUMN dosage_valid BOOLEAN",
     "Pre-compute result; index the boolean; instant query"),
]
for rank, strat, example, why in strategies:
    print(f"  {rank}. {strat}")
    print(f"     Example: {example}")
    print(f"     Why:     {why}")
    print()

print("Safety net -- always set statement_timeout for ad-hoc regex queries:")
print("  SET statement_timeout = '10s';  -- kills runaway patterns before they hurt DB")


---
## Common Pitfalls

**1. Regex for exact or simple prefix matches**
`WHERE mrn ~ '^MRN-'` should be `WHERE mrn LIKE 'MRN-%'` — LIKE with a constant prefix uses a B-tree index; the regex does not.

**2. Alternation regex instead of IN()**
`WHERE province ~ '^(NB|NS|PE|NL)$'` does a full sequential scan even on an indexed column. `WHERE province IN ('NB','NS','PE','NL')` uses the B-tree index.

**3. No pre-filter before free-text regex**
`WHERE note_text ~ 'lisinopril'` on 10M rows evaluates the regex 10M times. Always push the most selective indexed condition first.

**4. pg_trgm without ANALYZE**
Run `ANALYZE table_name` after creating a GIN trigram index so the planner knows it exists.

**5. Regex on LOWER(col) bypassing the index**
`WHERE LOWER(drug_name) ~ '^metformin'` cannot use a B-tree index on `drug_name`. Use `~*` or a functional index: `CREATE INDEX ON medications (LOWER(drug_name))`.

**6. No statement_timeout safety net**
A complex unanchored regex on a large table can run for minutes. Set `SET statement_timeout = '10s'` in sessions that run ad-hoc regex queries.


---
*sql_methods_library - Samantha McGarrigle*